# Resume Checker - ML Model Training & Analysis
This notebook trains an SVM model for resume classification and implements resume-job matching functionality using TF-IDF and Cosine Similarity.

In [16]:
# Import Required Libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import pickle
import textstat
from docx import Document
import PyPDF2
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

print("All libraries imported successfully!")

All libraries imported successfully!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load and Explore Resume Dataset
Load the resume dataset and perform exploratory data analysis.

In [17]:
# Load dataset - Resume.csv from root directory
try:
    print("Loading Resume.csv...")
    df = pd.read_csv('../../Resume.csv')
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head(2))
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    
    # Check if Resume_str column exists (actual column name in Resume.csv)
    if 'Resume_str' in df.columns:
        df['Resume'] = df['Resume_str']  # Create alias for compatibility
        print("\n✓ Using Resume_str column from CSV")
    elif 'Resume' not in df.columns:
        raise ValueError("Neither 'Resume' nor 'Resume_str' column found in CSV")
        
except FileNotFoundError:
    print("✗ Resume.csv not found at ../../Resume.csv")
    print("Make sure you're running the notebook from backend/notebooks/ directory")
    print("\nCreating sample dataset for demonstration...")
    # Create sample data if file doesn't exist
    df = pd.DataFrame({
        'Resume': [
            'Python Django Flask REST API development experience',
            'Java Spring Boot microservices architecture',
            'JavaScript React Node.js full stack developer',
            'Data Science Machine Learning scikit-learn pandas',
            'DevOps AWS Docker Kubernetes CI/CD'
        ] * 20,
        'Category': ['Software Engineer', 'Backend Developer', 'Frontend Developer', 'Data Scientist', 'DevOps Engineer'] * 20
    })
    print(f"Sample dataset created with shape: {df.shape}")

Loading Resume.csv...
Dataset loaded successfully!
Shape: (2484, 4)
Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']

First few rows:
         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  

Data types:
ID              int64
Resume_str     object
Resume_html    object
Category       object
dtype: object

Missing values:
ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

✓ Using Resume_str column from CSV


## Preprocess Resume Text Data
Clean and preprocess the resume text for feature extraction.

In [18]:
def preprocess_text(text):
    """
    Preprocess resume text with robust error handling
    """
    if not isinstance(text, str) or len(str(text).strip()) == 0:
        return ""
    
    try:
        # Convert to lowercase
        text = text.lower()
        
        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        
        # Remove email addresses
        text = re.sub(r'\S+@\S+', '', text)
        
        # Remove special characters and digits
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords
        stop_words = set(stopwords.words('english'))
        tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
        
        return ' '.join(tokens)
    except Exception as e:
        print(f"Warning: Error preprocessing text - {str(e)}")
        return ""

# Apply preprocessing
print("Preprocessing resume text...")
if 'Resume' in df.columns:
    # Remove any rows with null/NaN resume text
    print(f"Original dataset size: {len(df)}")
    df = df.dropna(subset=['Resume'])
    print(f"After removing nulls: {len(df)}")
    
    # Apply preprocessing with error handling
    df['Resume_cleaned'] = df['Resume'].apply(preprocess_text)
    
    # Remove rows where preprocessing resulted in empty text
    df = df[df['Resume_cleaned'].str.len() > 0]
    print(f"After removing empty texts: {len(df)}")
    
    print("✓ Text preprocessing completed!")
    print(f"\nSample preprocessed text (first 200 chars):")
    print(df['Resume_cleaned'].iloc[0][:200] if len(df) > 0 else "No data available")
else:
    print("✗ 'Resume' column not found in dataframe!")

Preprocessing resume text...
Original dataset size: 2484
After removing nulls: 2484
After removing empty texts: 2483
✓ Text preprocessing completed!

Sample preprocessed text (first 200 chars):
administratormarketing associate administrator summary dedicated customer service manager years experience hospitality customer service management respected builder leader customerfocused teams strive


## Feature Extraction with TF-IDF
Convert preprocessed text into TF-IDF feature vectors.

In [19]:
# Prepare data with validation
X = df['Resume_cleaned'] if 'Resume_cleaned' in df.columns else df['Resume']
y = df['Category'] if 'Category' in df.columns else df.iloc[:, -1]

print(f"Feature data shape: {X.shape}")
print(f"Target data shape: {y.shape}")
print(f"Unique categories: {y.nunique()}")
print(f"Category distribution:\n{y.value_counts()}\n")

# Validate sufficient data
if len(X) < 10:
    raise ValueError(f"Insufficient data: only {len(X)} samples. Need at least 10 for training.")

# Create TF-IDF Vectorizer with optimized parameters
print("Creating TF-IDF vectorizer...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,  # Limit features
    min_df=2,           # Ignore terms appearing in less than 2 documents
    max_df=0.8,         # Ignore terms appearing in more than 80% of documents
    stop_words='english'
)

X_tfidf = tfidf_vectorizer.fit_transform(X)

print(f"✓ TF-IDF matrix shape: {X_tfidf.shape}")
print(f"✓ Number of features: {len(tfidf_vectorizer.get_feature_names_out())}")
print(f"✓ Sparsity: {1.0 - (X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])):.2%}")
print(f"\nSample features: {tfidf_vectorizer.get_feature_names_out()[:10].tolist()}")

Feature data shape: (2483,)
Target data shape: (2483,)
Unique categories: 24
Category distribution:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      119
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

Creating TF-IDF vectorizer...
✓ TF-IDF matrix shape: (2483, 1000)
✓ Number of features: 1000
✓ Sparsity: 83.33%

Sample features: ['ability', 'a

## Train SVM Classification Model
Train an SVM classifier to categorize resumes by job type.

In [20]:
# Split data into training and testing sets
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X_tfidf, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError as e:
    # Handle case where stratified split is not possible (e.g., single class)
    print(f"Warning: Stratified split not possible ({e}), using random split instead")
    X_train, X_test, y_train, y_test = train_test_split(
        X_tfidf, y, test_size=0.2, random_state=42
    )

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Training set class distribution:\n{pd.Series(y_train).value_counts()}\n")

# Train SVM model
print("Training SVM model (this may take a moment)...")
svm_model = SVC(kernel='linear', probability=True, random_state=42, verbose=1)
svm_model.fit(X_train, y_train)
print("✓ SVM model trained successfully!")

# Make predictions
y_pred = svm_model.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"\n{'='*40}")
print(f"Model Performance Metrics")
print(f"{'='*40}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"{'='*40}")

Training set size: 1986 samples
Testing set size: 497 samples
Training set class distribution:
Category
INFORMATION-TECHNOLOGY    96
BUSINESS-DEVELOPMENT      95
FINANCE                   94
ACCOUNTANT                94
FITNESS                   94
CHEF                      94
ADVOCATE                  94
ENGINEERING               94
AVIATION                  93
SALES                     93
BANKING                   92
HEALTHCARE                92
CONSULTANT                92
CONSTRUCTION              90
PUBLIC-RELATIONS          89
HR                        88
DESIGNER                  86
TEACHER                   82
ARTS                      82
APPAREL                   78
DIGITAL-MEDIA             77
AGRICULTURE               50
AUTOMOBILE                29
BPO                       18
Name: count, dtype: int64

Training SVM model (this may take a moment)...
[LibSVM]✓ SVM model trained successfully!

Model Performance Metrics
Accuracy:  0.6680
Precision: 0.6733
Recall:    0.6680
F1-

## Parse Multiple File Formats (.txt, .docx, .pdf)
Implement functions to extract text from different file formats.

In [21]:
def parse_txt_file(file_path):
    """Extract text from .txt file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error reading TXT: {str(e)}"

def parse_docx_file(file_path):
    """Extract text from .docx file"""
    try:
        doc = Document(file_path)
        return '\n'.join([para.text for para in doc.paragraphs])
    except Exception as e:
        return f"Error reading DOCX: {str(e)}"

def parse_pdf_file(file_path):
    """Extract text from .pdf file"""
    try:
        text = ""
        with open(file_path, 'rb') as f:
            pdf_reader = PyPDF2.PdfReader(f)
            for page in pdf_reader.pages:
                text += page.extract_text()
        return text
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

print("File parsing functions defined successfully!")

File parsing functions defined successfully!


## Grammar and Readability Metrics
Calculate word count and Flesch-Kincaid readability score.

In [22]:
def calculate_readability_metrics(text):
    """
    Calculate grammar and readability metrics
    """
    # Word count
    words = text.split()
    word_count = len(words)
    
    # Sentence count
    sentences = text.split('.')
    sentence_count = len([s for s in sentences if s.strip()])
    
    # Flesch Kincaid Grade Level
    flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)
    
    # Flesch Reading Ease
    flesch_ease = textstat.flesch_reading_ease(text)
    
    # SMOG Index
    smog_index = textstat.smog_index(text)
    
    return {
        'word_count': word_count,
        'sentence_count': sentence_count,
        'flesch_reading_ease': round(flesch_ease, 2),
        'flesch_kincaid_grade': round(flesch_kincaid_grade, 2),
        'smog_index': round(smog_index, 2)
    }

# Test readability metrics
test_text = "I have extensive experience in Python programming and web development. My skills include Flask, Django, and REST API design."
metrics = calculate_readability_metrics(test_text)
print("=== Readability Metrics ===")
for key, value in metrics.items():
    print(f"{key}: {value}")

=== Readability Metrics ===
word_count: 19
sentence_count: 2
flesch_reading_ease: 50.26
flesch_kincaid_grade: 8.61
smog_index: 11.21


## Resume-Job Matching with Cosine Similarity
Implement cosine similarity-based matching between resumes and job descriptions.

In [23]:
def calculate_resume_job_match(resume_text, job_description, vectorizer):
    """
    Calculate cosine similarity between resume and job description
    """
    # Preprocess texts
    resume_clean = preprocess_text(resume_text)
    job_clean = preprocess_text(job_description)
    
    # Vectorize
    resume_vector = vectorizer.transform([resume_clean])
    job_vector = vectorizer.transform([job_clean])
    
    # Calculate cosine similarity
    similarity_score = cosine_similarity(resume_vector, job_vector)[0][0]
    
    return similarity_score * 100

# Test the matching function
test_resume = "Python Django Flask REST API SQL database development"
test_job = "Senior Python developer with Flask Django experience required"

match_score = calculate_resume_job_match(test_resume, test_job, tfidf_vectorizer)
print(f"Test Match Score: {match_score:.2f}%")

# Demonstrate matching with multiple resumes
print("\n=== Resume-Job Matching Demo ===")
sample_resumes = X.iloc[:3].tolist()
for i, resume in enumerate(sample_resumes):
    score = calculate_resume_job_match(resume, test_job, tfidf_vectorizer)
    print(f"Resume {i+1} match score: {score:.2f}%")

Test Match Score: 0.00%

=== Resume-Job Matching Demo ===
Resume 1 match score: 4.75%
Resume 2 match score: 2.19%
Resume 3 match score: 1.80%


## Skills Gap Analysis
Extract skills and identify gaps between resume and job requirements.

In [24]:
def extract_skills(text):
    """
    Extract technical skills from text
    """
    common_skills = {
        'programming': ['python', 'java', 'javascript', 'c++', 'c#', 'php', 'ruby', 'go', 'rust'],
        'web': ['html', 'css', 'react', 'angular', 'vue', 'node', 'express', 'flask', 'django'],
        'data': ['sql', 'mongodb', 'postgresql', 'mysql', 'elasticsearch', 'spark'],
        'cloud': ['aws', 'azure', 'gcp', 'docker', 'kubernetes', 'ci/cd', 'jenkins'],
        'data_science': ['pandas', 'numpy', 'sklearn', 'tensorflow', 'pytorch', 'ml'],
        'tools': ['git', 'linux', 'jira', 'confluence']
    }
    
    text_lower = text.lower()
    found_skills = []
    
    for category, skills in common_skills.items():
        for skill in skills:
            if skill in text_lower:
                found_skills.append(skill)
    
    return list(set(found_skills))

def analyze_skills_gap(resume_text, job_description):
    """
    Analyze skills gap between resume and job description
    """
    resume_skills = set(extract_skills(resume_text))
    job_skills = set(extract_skills(job_description))
    
    matched_skills = resume_skills & job_skills
    missing_skills = job_skills - resume_skills
    extra_skills = resume_skills - job_skills
    
    return {
        'matched_skills': list(matched_skills),
        'missing_skills': list(missing_skills),
        'extra_skills': list(extra_skills),
        'match_percentage': round(len(matched_skills) / max(1, len(job_skills)) * 100, 2)
    }

# Test skills gap analysis
gaps = analyze_skills_gap(test_resume, test_job)
print("=== Skills Gap Analysis ===")
print(f"Matched Skills: {gaps['matched_skills']}")
print(f"Missing Skills: {gaps['missing_skills']}")
print(f"Extra Skills: {gaps['extra_skills']}")
print(f"Match Percentage: {gaps['match_percentage']}%")

=== Skills Gap Analysis ===
Matched Skills: ['go', 'python', 'flask', 'django']
Missing Skills: []
Extra Skills: ['sql']
Match Percentage: 100.0%


## Comprehensive Resume Analysis Function
Create a complete function that performs all analysis steps.

In [25]:
def analyze_resume_comprehensive(resume_text, job_description):
    """
    Perform comprehensive resume analysis
    """
    # Grammar & Readability
    readability = calculate_readability_metrics(resume_text)
    
    # Keyword Match
    resume_clean = preprocess_text(resume_text)
    job_clean = preprocess_text(job_description)
    match_score = calculate_resume_job_match(resume_text, job_description, tfidf_vectorizer)
    
    # Skills Gap
    skills_gap = analyze_skills_gap(resume_text, job_description)
    
    # Combine all metrics
    comprehensive_analysis = {
        'grammar_readability': readability,
        'keyword_match': {
            'score': round(match_score, 2),
            'level': 'Excellent' if match_score > 70 else 'Good' if match_score > 50 else 'Fair'
        },
        'skills_gap': skills_gap,
        'overall_match': round((match_score + skills_gap['match_percentage']) / 2, 2)
    }
    
    return comprehensive_analysis

# Test comprehensive analysis
print("=== Comprehensive Resume Analysis ===\n")
analysis = analyze_resume_comprehensive(test_resume, test_job)
for category, result in analysis.items():
    print(f"{category}: {result}\n")

=== Comprehensive Resume Analysis ===

grammar_readability: {'word_count': 8, 'sentence_count': 1, 'flesch_reading_ease': 40.09, 'flesch_kincaid_grade': 9.66, 'smog_index': 11.21}

keyword_match: {'score': np.float64(0.0), 'level': 'Fair'}

skills_gap: {'matched_skills': ['go', 'python', 'flask', 'django'], 'missing_skills': [], 'extra_skills': ['sql'], 'match_percentage': 100.0}

overall_match: 50.0



## Save Models and Vectorizer
Save trained models for use in Flask API.

In [26]:
# Save the trained models
import os

print("Saving trained models...")

# Create models directory if it doesn't exist
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

try:
    # Save TF-IDF vectorizer
    tfidf_path = os.path.join(models_dir, 'tfidf_vectorizer.pkl')
    with open(tfidf_path, 'wb') as f:
        pickle.dump(tfidf_vectorizer, f)
    print(f"✓ TF-IDF vectorizer saved to {tfidf_path}")
    
    # Save SVM model
    model_path = os.path.join(models_dir, 'svm_model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(svm_model, f)
    print(f"✓ SVM model saved to {model_path}")
    
    # Verify models were saved
    if os.path.exists(tfidf_path) and os.path.exists(model_path):
        print(f"\n{'='*50}")
        print(f"All models saved successfully to {models_dir}/ directory!")
        print(f"{'='*50}")
        print(f"Files created:")
        print(f"  - tfidf_vectorizer.pkl ({os.path.getsize(tfidf_path) / 1024 / 1024:.2f} MB)")
        print(f"  - svm_model.pkl ({os.path.getsize(model_path) / 1024 / 1024:.2f} MB)")
    else:
        print("✗ Error: Models were not saved correctly!")
        
except Exception as e:
    print(f"✗ Error saving models: {str(e)}")
    raise

Saving trained models...
✓ TF-IDF vectorizer saved to ../models\tfidf_vectorizer.pkl
✓ SVM model saved to ../models\svm_model.pkl

All models saved successfully to ../models/ directory!
Files created:
  - tfidf_vectorizer.pkl (0.04 MB)
  - svm_model.pkl (3.74 MB)
